# LeetCode 461 — Hamming Distance bằng Python

**Link đề:** https://leetcode.com/problems/hamming-distance/
**Độ khó:** Easy · **Chủ đề:** Bit Manipulation (thao tác bit)

```python
class Solution:
    def hammingDistance(self, x: int, y: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **khoảng cách Hamming**: số vị trí bit mà hai số khác nhau.
2. Làm quen hệ nhị phân, `bin()`, `format(n, '032b')`, các toán tử bit `&`, `>>`, và đặc biệt **XOR `^`**.
3. Đi từ so sánh hai chuỗi bit → XOR rồi đếm bit 1 (bằng chuỗi, bằng vòng lặp, bằng mẹo Brian Kernighan `n & (n-1)`, bằng `int.bit_count()`).
4. Trace từng bước, chứng minh "XOR đánh dấu đúng các vị trí khác nhau", phân tích Big-O.
5. Nhận dạng pattern "so sánh bit → XOR", làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

**Khoảng cách Hamming** giữa hai số nguyên là số vị trí mà bit tương ứng của chúng **khác nhau**. Cho `x`, `y`, trả về khoảng cách Hamming.

| `x` | `y` | nhị phân `x` | nhị phân `y` | Output |
|---:|---:|---|---|:---:|
| 1 | 4 | `0001` | `0100` | 2 |
| 3 | 1 | `0011` | `0001` | 1 |
| 0 | 0 | `0000` | `0000` | 0 |

Ví dụ 1 trong đề:
```text
1   (0 0 0 1)
4   (0 1 0 0)
       ↑   ↑     <- hai vị trí khác nhau
```

**Constraints:** `0 <= x, y <= 2^31 - 1` (số không âm, vừa 31 bit).

Ghi chú: bài này giống hệt LeetCode 2220 (Minimum Bit Flips to Convert Number) — số bit cần lật để biến `x` thành `y` chính là số vị trí khác nhau.

## 2. Nền tảng — hệ nhị phân

Hệ nhị phân chỉ có hai chữ số `0` và `1`; mỗi vị trí mạnh gấp **2** lần vị trí bên phải. Mỗi chữ số gọi là một **bit**. Bit bên phải nhất (vị trí 0) là bit thấp nhất.

| vị trí | 3 | 2 | 1 | 0 |
|---|---|---|---|---|
| giá trị vị trí | 8 | 4 | 2 | 1 |
| bit của 4 | 0 | 1 | 0 | 0 |
| bit của 1 | 0 | 0 | 0 | 1 |

`4 = 1·4`, `1 = 1·1`. Hai số khác nhau ở vị trí 0 và vị trí 2 → khoảng cách 2.

In [1]:
# Đổi số sang list bit thủ công (từ vị trí thấp đến cao) bằng % 2 và // 2
def to_bits(n, width=4):
    bits = []
    for _ in range(width):
        bits.append(n % 2)   # bit thấp nhất
        n //= 2              # bỏ bit thấp nhất
    return bits[::-1]        # đảo lại để đọc từ trái sang phải

print("1 ->", to_bits(1))
print("4 ->", to_bits(4))
print("11 ->", to_bits(11))

1 -> [0, 0, 0, 1]
4 -> [0, 1, 0, 0]
11 -> [1, 0, 1, 1]


### 2.1. `bin()` và `format()` — xem bit trong Python

- `bin(n)` trả về chuỗi có tiền tố `'0b'`: `bin(4) == '0b100'`.
- `format(n, 'b')` trả chuỗi bit không tiền tố; `format(n, '032b')` **đệm 0 bên trái** cho đủ 32 ký tự.
- Đệm bằng nhau rất quan trọng khi so sánh hai số có độ dài bit khác nhau (`1` có 1 bit, `4` có 3 bit).

In [2]:
print(bin(4), bin(1))
print(format(4, 'b'), format(1, 'b'))
print(format(4, '08b'))
print(format(1, '08b'))
print("độ dài khác nhau nếu không đệm:", len(format(4, 'b')), len(format(1, 'b')))

0b100 0b1
100 1
00000100
00000001
độ dài khác nhau nếu không đệm: 3 1


### 2.2. Toán tử `&` (AND) và `>>` (dịch phải)

- `a & b`: bit kết quả là 1 khi **cả hai** bit đều 1. `n & 1` lấy **bit cuối** của `n`.
- `n >> 1`: dịch các bit sang phải 1 vị trí, bit cuối rơi mất — bằng `n // 2` với số không âm.

In [3]:
n = 11  # 1011
print(f"{n:04b} & 0001 = {n & 1}")
print(f"{n:04b} >> 1  = {n >> 1:04b} ({n >> 1})")
print(f"0b1100 & 0b1010 = {0b1100 & 0b1010:04b}")

1011 & 0001 = 1
1011 >> 1  = 0101 (5)
0b1100 & 0b1010 = 1000


Giải thích `f"{n:04b}"`: trong f-string, sau dấu `:` là định dạng — `b` là nhị phân, `04` là đệm 0 cho đủ 4 ký tự. `0b1100` là cách viết **literal nhị phân** trực tiếp trong code (bằng 12).

### 2.3. Toán tử XOR `^` — trái tim của bài này

XOR (exclusive or — "hoặc loại trừ"): bit kết quả là **1 khi hai bit khác nhau**, 0 khi giống nhau.

| a | b | a ^ b |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Vậy `x ^ y` có bit 1 **đúng tại các vị trí x và y khác nhau**. Khoảng cách Hamming = **số bit 1 của `x ^ y`**.

In [4]:
x, y = 1, 4
print(f"x     = {x:04b}")
print(f"y     = {y:04b}")
print(f"x ^ y = {x ^ y:04b}  -> có {bin(x ^ y).count('1')} bit 1")
# Một vài tính chất của XOR
print("a ^ a =", 7 ^ 7, "| a ^ 0 =", 7 ^ 0, "| giao hoán:", 5 ^ 9 == 9 ^ 5)

x     = 0001
y     = 0100
x ^ y = 0101  -> có 2 bit 1
a ^ a = 0 | a ^ 0 = 7 | giao hoán: True


### 2.4. `str.count()` — đếm ký tự

`"1011".count("1") == 3`. Tiền tố `'0b'` không chứa ký tự `'1'` nên `bin(n).count('1')` đếm đúng số bit 1.

## 3. Cách 1 — Trực giác: viết ra hai chuỗi bit rồi so từng vị trí

Người mới thường làm đúng như ví dụ của đề: viết hai số dưới dạng nhị phân **cùng độ dài** (32 bit), rồi đếm vị trí khác nhau. Dùng `zip(a, b)` để đi song song hai chuỗi: `zip("abc", "xyz")` sinh các cặp `('a','x'), ('b','y'), ('c','z')`.

In [5]:
print(list(zip("0001", "0100")))

def hamming_string_compare(x: int, y: int) -> int:
    bx = format(x, '032b')
    by = format(y, '032b')
    count = 0
    for cx, cy in zip(bx, by):
        if cx != cy:
            count += 1
    return count

print(hamming_string_compare(1, 4))   # 2
print(hamming_string_compare(3, 1))   # 1

[('0', '0'), ('0', '1'), ('0', '0'), ('1', '0')]
2
1


Cách này đúng và dễ hiểu, nhưng: tạo hai chuỗi 32 ký tự, so 32 lần **kể cả** khi hai số giống nhau gần hết. Quan trọng hơn: ta đang "tự làm thủ công" việc mà phần cứng làm sẵn trong một lệnh — **XOR**.

## 4. Bước đột phá — XOR biến "so sánh" thành "đếm"

> `hamming(x, y) = popcount(x ^ y)` (popcount = số bit 1).

Bài toán hai số được rút gọn thành bài toán **một số**: đếm bit 1 (chính là LeetCode 191). Từ đây có nhiều cách đếm.

In [6]:
def hamming_xor_str(x: int, y: int) -> int:
    return bin(x ^ y).count('1')

for x, y in [(1, 4), (3, 1), (0, 0), (2**31 - 1, 0)]:
    print(f"x={x}, y={y} -> x^y={x ^ y:b} -> {hamming_xor_str(x, y)}")

x=1, y=4 -> x^y=101 -> 2
x=3, y=1 -> x^y=10 -> 1
x=0, y=0 -> x^y=0 -> 0
x=2147483647, y=0 -> x^y=1111111111111111111111111111111 -> 31


## 5. Lời giải chính — XOR rồi đếm bit 1 bằng vòng lặp (mổ xẻ từng đoạn)

Ta muốn tự đếm mà không dùng chuỗi, để hiểu thao tác bit.

### 5.1. Đoạn 1: tính `diff = x ^ y`
### 5.2. Đoạn 2: đọc bit cuối `diff & 1`, cộng vào `count`
### 5.3. Đoạn 3: bỏ bit cuối `diff >>= 1`; lặp tới khi `diff == 0`

Vòng lặp dừng khi không còn bit 1 nào phía trước → số vòng = số bit có nghĩa của `diff` (≤ 31).

In [7]:
def hamming_shift(x: int, y: int) -> int:
    diff = x ^ y
    count = 0
    while diff:              # diff != 0
        count += diff & 1    # cộng bit cuối (0 hoặc 1)
        diff >>= 1           # bỏ bit cuối
    return count

print(hamming_shift(1, 4), hamming_shift(3, 1), hamming_shift(0, 0))

2 1 0


`while diff:` — trong Python, số `0` được coi là `False`, số khác 0 là `True` (truthiness). Viết `while diff:` tương đương `while diff != 0:`.

## 6. TRACE từng bước — `x = 1`, `y = 4`

`diff = 1 ^ 4 = 0b101 = 5`.

| vòng | `diff` (bin) | `diff & 1` | `count` sau | `diff >> 1` |
|---|---|---|---|---|
| 1 | `101` | 1 | 1 | `10` |
| 2 | `10` | 0 | 1 | `1` |
| 3 | `1` | 1 | 2 | `0` |
| — | `0` | dừng | **2** | |

In [8]:
def trace_hamming_shift(x, y):
    diff = x ^ y
    print(f"x = {x:b}, y = {y:b}, diff = x ^ y = {diff:b}")
    count = 0
    step = 1
    while diff:
        bit = diff & 1
        count += bit
        print(f"vòng {step}: diff={diff:>6b}, bit cuối={bit}, count={count}, diff>>1={diff >> 1:b}")
        diff >>= 1
        step += 1
    print("Kết quả:", count)
    return count

trace_hamming_shift(1, 4)
print()
trace_hamming_shift(93, 73)   # nhiều bit hơn

x = 1, y = 100, diff = x ^ y = 101
vòng 1: diff=   101, bit cuối=1, count=1, diff>>1=10
vòng 2: diff=    10, bit cuối=0, count=1, diff>>1=1
vòng 3: diff=     1, bit cuối=1, count=2, diff>>1=0
Kết quả: 2

x = 1011101, y = 1001001, diff = x ^ y = 10100
vòng 1: diff= 10100, bit cuối=0, count=0, diff>>1=1010
vòng 2: diff=  1010, bit cuối=0, count=0, diff>>1=101
vòng 3: diff=   101, bit cuối=1, count=1, diff>>1=10
vòng 4: diff=    10, bit cuối=0, count=1, diff>>1=1
vòng 5: diff=     1, bit cuối=1, count=2, diff>>1=0
Kết quả: 2


2

## 7. Cách 3 — Mẹo Brian Kernighan: `n & (n - 1)` xóa bit 1 thấp nhất

Trừ 1 làm bit 1 thấp nhất của `n` thành 0 và tất cả bit 0 phía sau nó thành 1. AND với `n` gốc thì **xóa đúng bit 1 thấp nhất**:

```text
n       = 1011000
n - 1   = 1010111
n&(n-1) = 1010000   <- bit 1 ở vị trí 3 đã biến mất
```

Mỗi vòng xóa một bit 1 → số vòng = **số bit 1** (không phải tổng số bit). Khi hai số gần giống nhau (ít bit khác), cách này rất nhanh.

In [9]:
n = 0b1011000
print(f"n       = {n:07b}")
print(f"n - 1   = {n - 1:07b}")
print(f"n&(n-1) = {n & (n - 1):07b}")

def hamming_kernighan(x: int, y: int) -> int:
    diff = x ^ y
    count = 0
    while diff:
        diff &= diff - 1     # xóa bit 1 thấp nhất
        count += 1
    return count

print(hamming_kernighan(1, 4), hamming_kernighan(3, 1))

n       = 1011000
n - 1   = 1010111
n&(n-1) = 1010000
2 1


In [10]:
def trace_kernighan(x, y):
    diff = x ^ y
    print(f"diff = {diff:b}")
    count = 0
    while diff:
        new = diff & (diff - 1)
        count += 1
        print(f"  {diff:>8b} & {diff - 1:>8b} = {new:>8b}  -> đã xóa 1 bit, count = {count}")
        diff = new
    return count

trace_kernighan(93, 73)

diff = 10100
     10100 &    10011 =    10000  -> đã xóa 1 bit, count = 1
     10000 &     1111 =        0  -> đã xóa 1 bit, count = 2


2

## 8. Cách 4 — Hàm có sẵn `int.bit_count()` (Python 3.10+)

`n.bit_count()` trả số bit 1 của `n` (popcount) bằng mã C rất nhanh. Trên LeetCode (Python 3 hiện đại) dùng được; nhưng phỏng vấn thường muốn bạn hiểu cách tự đếm.

In [11]:
print((5).bit_count(), (0).bit_count(), (2**31 - 1).bit_count())

def hamming_builtin(x: int, y: int) -> int:
    return (x ^ y).bit_count()

print(hamming_builtin(1, 4), hamming_builtin(3, 1))

2 0 31
2 1


Chú ý cú pháp `(5).bit_count()`: phải có ngoặc, vì `5.bit_count()` bị Python hiểu `5.` là số thực rồi báo lỗi cú pháp. Với biến thì viết bình thường: `diff.bit_count()`.

## 9. Code nộp LeetCode (Python 3)

Bản chính: XOR + Kernighan (thể hiện hiểu biết về bit, số vòng = số bit khác nhau).

In [12]:
class Solution:
    def hammingDistance(self, x: int, y: int) -> int:
        diff = x ^ y
        count = 0
        while diff:
            diff &= diff - 1
            count += 1
        return count

sol = Solution()
print(sol.hammingDistance(1, 4))   # 2
print(sol.hammingDistance(3, 1))   # 1

2
1


Bản ngắn gọn (cũng Accepted):

```python
class Solution:
    def hammingDistance(self, x: int, y: int) -> int:
        return bin(x ^ y).count('1')      # hoặc (x ^ y).bit_count()
```

## 10. Vì sao đúng?

1. **XOR đúng vị trí:** theo bảng chân trị, bit thứ `k` của `x ^ y` bằng 1 ⇔ bit thứ `k` của `x` khác bit thứ `k` của `y`. Vậy số bit 1 của `x ^ y` = số vị trí khác nhau. Các vị trí cao hơn bit cao nhất của cả hai số đều là 0 ở cả hai → XOR là 0 → không đếm nhầm (đây là lý do không cần "đệm" khi dùng XOR).
2. **Vòng dịch phải** — invariant: `count` + (số bit 1 của `diff` hiện tại) = số bit 1 của `x ^ y` ban đầu. Mỗi vòng chuyển bit cuối từ `diff` sang `count`; khi `diff = 0` thì `count` là đáp án.
3. **Kernighan** — invariant: `count` + popcount(`diff`) không đổi; mỗi vòng `diff & (diff-1)` giảm popcount đúng 1 và `count` tăng 1.

## 11. Phân tích Time / Space

Gọi `B` = số bit của số (ở đây `B <= 31`), `k` = số bit khác nhau (`k <= B`).

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. So hai chuỗi 32 bit | `O(B)` | `O(B)` | trực giác, tạo chuỗi |
| 2. `bin(x^y).count('1')` | `O(B)` | `O(B)` | một dòng |
| 3. XOR + dịch phải | `O(B)` | `O(1)` | hiểu bit |
| 4. XOR + Kernighan | `O(k)` | `O(1)` | số vòng = số bit khác |
| 5. `(x^y).bit_count()` | `O(B)` (C, rất nhanh) | `O(1)` | Python 3.10+ |

Vì `B` bị chặn bởi 31, mọi cách thực chất là `O(1)` theo kích thước input; bảng trên giúp so sánh khi số bit tăng.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

MAX = 2**31 - 1
cases = [
    ("đề 1", 1, 4, 2),
    ("đề 2", 3, 1, 1),
    ("hai số 0", 0, 0, 0),
    ("hai số bằng nhau", 12345, 12345, 0),
    ("0 và max", 0, MAX, 31),
    ("max và max", MAX, MAX, 0),
    ("lũy thừa 2 khác nhau", 1, 2**30, 2),
    ("bù trong 4 bit", 0b1010, 0b0101, 4),
]
sol = Solution()
for name, x, y, expected in cases:
    check(name, sol.hammingDistance(x, y), expected)

PASS  đề 1: 2
PASS  đề 2: 1
PASS  hai số 0: 0
PASS  hai số bằng nhau: 0
PASS  0 và max: 31
PASS  max và max: 0
PASS  lũy thừa 2 khác nhau: 2
PASS  bù trong 4 bit: 4


In [14]:
import random
random.seed(461)
methods = [hamming_string_compare, hamming_xor_str, hamming_shift, hamming_kernighan, hamming_builtin]
for _ in range(2000):
    x = random.randint(0, MAX)
    y = random.randint(0, MAX)
    results = {f(x, y) for f in methods}
    assert len(results) == 1, (x, y, results)
    # tính chất: đối xứng và bất đẳng thức tam giác
    z = random.randint(0, MAX)
    assert hamming_builtin(x, y) == hamming_builtin(y, x)
    assert hamming_builtin(x, z) <= hamming_builtin(x, y) + hamming_builtin(y, z)
print("2000 cặp ngẫu nhiên: 5 cách khớp; đối xứng và bất đẳng thức tam giác đúng")

2000 cặp ngẫu nhiên: 5 cách khớp; đối xứng và bất đẳng thức tam giác đúng


Khoảng cách Hamming là một **metric** (khoảng cách đúng nghĩa toán học): `d(x,y) = 0` ⇔ `x = y`, đối xứng, và thỏa bất đẳng thức tam giác. Test trên kiểm chứng hai tính chất cuối.

## 13. Lỗi người mới hay mắc

### Lỗi 1: so hai chuỗi `bin()` **không đệm** độ dài
`bin(1) = '0b1'`, `bin(4) = '0b100'` — `zip` sẽ dừng ở chuỗi ngắn hơn và ghép sai vị trí (vị trí trái nhất của chuỗi này ứng với vị trí khác của chuỗi kia).

In [15]:
def wrong_no_padding(x, y):
    bx, by = bin(x)[2:], bin(y)[2:]
    return sum(1 for a, c in zip(bx, by) if a != c)

print("sai :", wrong_no_padding(1, 4))    # chỉ so 1 ký tự '1' với '1'
print("đúng:", hamming_xor_str(1, 4))

sai : 0
đúng: 2


### Lỗi 2: dùng `|` (OR) hoặc `&` (AND) thay vì `^`
OR đánh dấu vị trí "ít nhất một bên là 1", AND đánh dấu "cả hai là 1" — đều không phải "khác nhau".

In [16]:
x, y = 3, 1   # 11 và 01
print("OR :", bin(x | y).count('1'), "| AND:", bin(x & y).count('1'), "| XOR (đúng):", bin(x ^ y).count('1'))

OR : 2 | AND: 1 | XOR (đúng): 1


### Lỗi 3: nhầm `^` là lũy thừa
Trong Python, lũy thừa là `**`. `2 ^ 3 == 1` (XOR), còn `2 ** 3 == 8`. Lỗi này rất hay gặp khi đọc đề viết `2^31`.

### Lỗi 4: thứ tự ưu tiên toán tử
Trong Python, thứ tự ưu tiên từ cao xuống thấp là: `-` (trừ) → `&` → `^` → `|` → các phép so sánh (`==`, `<`...). Do đó:
- `diff & diff - 1` được hiểu là `diff & (diff - 1)` — đúng ý.
- `diff & 1 == 1` được hiểu là `(diff & 1) == 1` — cũng đúng ý, **nhưng khác ngôn ngữ C** (ở C, `==` ưu tiên cao hơn `&`).

Để an toàn khi chuyển qua lại giữa các ngôn ngữ và để dễ đọc: **luôn thêm ngoặc** quanh biểu thức bit.

In [17]:
print("2 ^ 3  =", 2 ^ 3, "| 2 ** 3 =", 2 ** 3)
diff = 6
print(diff & diff - 1, "==", diff & (diff - 1))
print((diff & 1) == 1, "==", diff & 1 == 1)   # Python: & ưu tiên cao hơn ==, kết quả như nhau

2 ^ 3  = 1 | 2 ** 3 = 8
4 == 4
False == False


### Lỗi 5: vòng lặp cố định 32 lần nhưng dùng số âm
Đề bảo đảm số không âm. Nếu tự mở rộng cho số âm, `-1 >> 1` trong Python vẫn là `-1` (dịch số học, số nguyên không giới hạn bit) → `while diff:` **lặp vô hạn**. Cần mặt nạ `diff &= 0xFFFFFFFF` nếu muốn mô phỏng 32 bit. Ta chỉ minh họa an toàn bằng số vòng giới hạn.

In [18]:
d = -1
for _ in range(3):
    d >>= 1
    print("(-1) >> 1 vẫn là", d)
print("Mặt nạ 32 bit của -1:", format(-1 & 0xFFFFFFFF, 'b'), "->", (-1 & 0xFFFFFFFF).bit_count(), "bit 1")

(-1) >> 1 vẫn là -1
(-1) >> 1 vẫn là -1
(-1) >> 1 vẫn là -1
Mặt nạ 32 bit của -1: 11111111111111111111111111111111 -> 32 bit 1


## 14. Mẹo nhận dạng pattern

- "Hai số **khác nhau ở bit nào**?" / "lật bao nhiêu bit để biến a thành b?" → **XOR**.
- "Đếm bit 1" → `bin().count('1')`, vòng `& 1` + `>> 1`, Kernighan `n & (n-1)`, hoặc `bit_count()`.
- XOR còn dùng để **khử cặp trùng** (`a ^ a = 0`): Single Number (136), Missing Number (268).

Bài cùng họ: 191 Number of 1 Bits, 338 Counting Bits, 477 Total Hamming Distance, 2220 Minimum Bit Flips, 136 Single Number, 476 Number Complement.

## 15. Biến thể

### 15.1. Total Hamming Distance (LeetCode 477) — tổng khoảng cách mọi cặp

Cách ngây thơ: mọi cặp `O(n²)`. Mẹo: xét **từng vị trí bit** độc lập; nếu có `c` số có bit 1 và `n - c` số có bit 0 ở vị trí đó thì vị trí này đóng góp `c · (n - c)` cặp khác nhau.

In [19]:
def total_hamming_naive(nums):
    return sum(hamming_builtin(nums[i], nums[j]) for i in range(len(nums)) for j in range(i + 1, len(nums)))

def total_hamming_by_bit(nums):
    n = len(nums)
    total = 0
    for bit in range(31):
        c = sum((v >> bit) & 1 for v in nums)
        total += c * (n - c)
    return total

nums = [4, 14, 2]
print(total_hamming_naive(nums), total_hamming_by_bit(nums))   # 6 6

6 6


### 15.2. Khoảng cách Hamming giữa hai **chuỗi** cùng độ dài

In [20]:
def hamming_strings(a: str, b_: str) -> int:
    assert len(a) == len(b_)
    return sum(c1 != c2 for c1, c2 in zip(a, b_))

print(hamming_strings("karolin", "kathrin"))   # 3

3


`sum(c1 != c2 for ...)`: `True` được cộng như `1`, `False` như `0` — mẹo đếm rất gọn.

## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Gợi ý ở cuối mục này, đáp án ở mục 17.

### Bài 1 — Dự đoán output
Không chạy code: `6 ^ 3` bằng bao nhiêu? Khoảng cách Hamming giữa 6 và 3 là bao nhiêu?

### Bài 2 — Tính tay Kernighan
Với `diff = 0b10110`, ghi giá trị `diff` sau mỗi bước `diff &= diff - 1` cho tới 0. Có bao nhiêu bước?

### Bài 3 — Hoàn thiện cách dịch phải

In [21]:
def ex3_hamming(x: int, y: int) -> int:
    # TODO 1: diff = x ^ y
    # TODO 2: while diff: cộng diff & 1 vào count, rồi diff >>= 1
    pass

# assert ex3_hamming(1, 4) == 2
# assert ex3_hamming(3, 1) == 1

### Bài 4 — Minimum Bit Flips trong một list
Cho `start` và list `goals`, trả về list số bit cần lật để biến `start` thành từng `goal`. Ví dụ `start = 10`, `goals = [7, 10, 0]` → `[3, 0, 2]`.

In [22]:
def ex4_bit_flips(start: int, goals: list[int]) -> list[int]:
    # TODO
    pass

# assert ex4_bit_flips(10, [7, 10, 0]) == [3, 0, 2]

### Bài 5 — Mini challenge: Total Hamming Distance không dùng hai vòng lặp cặp
Viết lại `total_hamming` theo từng bit **mà không** nhìn code ở mục 15.1, rồi kiểm tra với `[4, 14, 4]` → `4`.

In [23]:
def ex5_total_hamming(nums: list[int]) -> int:
    # TODO: với mỗi bit 0..30 đếm số phần tử có bit 1
    pass

# assert ex5_total_hamming([4, 14, 2]) == 6
# assert ex5_total_hamming([4, 14, 4]) == 4

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: viết `6 = 110`, `3 = 011`. Mức 2: XOR từng cột. Mức 3: `101`.

**Bài 2** — Mức 1: mỗi bước xóa bit 1 phải nhất. Mức 2: `10110 → 10100`. Mức 3: số bước = số bit 1.

**Bài 3** — Mức 1: XOR trước. Mức 2: `count += diff & 1`. Mức 3: đừng quên `diff >>= 1`, nếu không sẽ lặp vô hạn.

**Bài 4** — Mức 1: mỗi goal là một bài 461. Mức 2: list comprehension. Mức 3: `[(start ^ g).bit_count() for g in goals]`.

**Bài 5** — Mức 1: `(v >> bit) & 1` lấy bit thứ `bit`. Mức 2: đếm `c` số có bit 1. Mức 3: cộng `c * (n - c)`.

## 17. Đáp án tham khảo

**Bài 1:** `6 ^ 3 = 0b101 = 5`, khoảng cách = 2.

**Bài 2:** `10110 → 10100 → 10000 → 0` — 3 bước (bằng số bit 1).

In [24]:
def ans3_hamming(x, y):
    diff = x ^ y
    count = 0
    while diff:
        count += diff & 1
        diff >>= 1
    return count

def ans4_bit_flips(start, goals):
    return [(start ^ g).bit_count() for g in goals]

def ans5_total_hamming(nums):
    n = len(nums)
    total = 0
    for bit in range(31):
        c = sum((v >> bit) & 1 for v in nums)
        total += c * (n - c)
    return total

assert 6 ^ 3 == 5 and ans3_hamming(6, 3) == 2            # bài 1
steps = []
d = 0b10110
while d:
    d &= d - 1
    steps.append(format(d, 'b'))
assert steps == ['10100', '10000', '0']                  # bài 2
assert ans3_hamming(1, 4) == 2 and ans3_hamming(3, 1) == 1
assert ans4_bit_flips(10, [7, 10, 0]) == [3, 0, 2]
assert ans5_total_hamming([4, 14, 2]) == 6
assert ans5_total_hamming([4, 14, 4]) == 4
print("Đáp án bài 1–5: PASS")

Đáp án bài 1–5: PASS


## 18. Template nhớ nhanh + checklist submit

```python
diff = x ^ y            # bit 1 = vị trí khác nhau
count = 0
while diff:
    diff &= diff - 1    # xóa bit 1 thấp nhất
    count += 1
return count
# hoặc: return bin(x ^ y).count('1')   /   (x ^ y).bit_count()
```

**Checklist trước khi submit**
- [ ] Dùng `^` (XOR), không nhầm với `|`, `&` hay lũy thừa `**`?
- [ ] Nếu so chuỗi bit, đã đệm cùng độ dài (`'032b'`) chưa?
- [ ] Vòng `while` có làm `diff` giảm (dịch phải hoặc xóa bit) chưa?
- [ ] Ca `x == y` trả `0`?
- [ ] Nói được vì sao popcount(`x ^ y`) đúng bằng số vị trí khác nhau?